# Credit Risk Decisioning System

End-to-end recruiter-grade workflow with four modeling tracks (LazyPredict, Manual Top-3, FLAML, PyCaret).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src.data_prep import load_home_credit_data, build_customer_level_table, temporal_leakage_checks, stratified_split
from src.features import build_preprocessor, prepare_model_inputs
from src.modeling import (
    run_lazypredict_discovery,
    select_top3_eligible_families,
    run_manual_engineering_track,
    run_flaml_track,
    run_pycaret_track,
    assign_decision_band,
    save_inference_bundle,
)
from src.evaluation import build_leaderboard, rerank_with_business_weights

SEED = 42
PROJECT_NAME = 'credit-risk-decisioning-system'
PROJECT_ROOT = Path('.')
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'home_credit'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Business Problem and Success Criteria

Goal: reduce default risk while preserving approval volume.

Success criteria:
- Strong PR-AUC (class imbalance aware)
- Stable ROC-AUC and Brier calibration
- Actionable approve/review/reject policy
- Deployable scoring and monitoring hooks

## 2) Dataset Access and Data Dictionary

Dataset: Home Credit Default Risk (Kaggle).

Primary table:
- `application_train.csv`

Auxiliary tables used for aggregated customer signals:
- `bureau.csv`
- `previous_application.csv`
- `POS_CASH_balance.csv`
- `installments_payments.csv`
- `credit_card_balance.csv`

In [ ]:
tables = load_home_credit_data(RAW_DIR, sample_frac=0.05, random_state=SEED)
{k: v.shape for k, v in tables.items()}

## 3) Data Cleaning and Leakage Checks

- Build customer-level table from auxiliary aggregates
- Remove leak-prone or near-empty fields
- Validate target integrity before split

In [ ]:
customer_df = build_customer_level_table(tables)
customer_df = temporal_leakage_checks(customer_df)
customer_df['TARGET'].value_counts(normalize=True)

## 4) Feature Engineering

- Numeric aggregation features from auxiliary tables
- Categorical one-hot encoding
- Median/mode imputations and scaling for numeric stability

In [ ]:
train_df, holdout_df = stratified_split(customer_df, target_col='TARGET', random_state=SEED)
preprocessor = build_preprocessor(train_df.drop(columns=['TARGET']))
X_train, X_holdout, y_train, y_holdout = prepare_model_inputs(
    train_df, holdout_df, target_col='TARGET', preprocessor=preprocessor
)
X_train.shape, X_holdout.shape

## 5) Validation Strategy

- Stratified train/holdout split
- Track-specific CV where applicable (manual, FLAML, PyCaret)
- Primary metric: PR-AUC
- Secondary metric: ROC-AUC
- Tertiary metric: Brier score

In [ ]:
summary = pd.DataFrame({
    'split': ['train', 'holdout'],
    'rows': [len(train_df), len(holdout_df)],
    'target_rate': [train_df['TARGET'].mean(), holdout_df['TARGET'].mean()]
})
summary

## 6) Track 1: LazyPredict Discovery Lab

LazyPredict runs on the post-preprocessing matrix to identify promising model families quickly.

In [ ]:
lazy_table = run_lazypredict_discovery(X_train, X_holdout, y_train, y_holdout)
lazy_table.head(15)

## 7) Selection of Top 3 Eligible Models

Eligibility filters applied:
- stability (Brier not extreme)
- speed and practicality
- interpretability fit for risk use-case

Only eligible top 3 families move into manual engineering.

In [ ]:
eligible_table, top3_families = select_top3_eligible_families(
    lazy_table, X_train, y_train, X_holdout, y_holdout, random_state=SEED
)
eligible_table, top3_families

## 8) Track 2: Manual Engineering Lab

For each selected family:
- explicit training code
- calibration where relevant
- threshold optimization
- decision-band assignment

In [ ]:
manual_results, manual_models = run_manual_engineering_track(
    top3_families=top3_families,
    X_train=X_train, y_train=y_train,
    X_holdout=X_holdout, y_holdout=y_holdout,
    random_state=SEED,
)
manual_results[['model_name', 'pr_auc', 'roc_auc', 'brier', 'optimized_threshold', 'policy_utility']]

In [ ]:
best_manual_idx = manual_results['pr_auc'].idxmax()
best_manual_row = manual_results.loc[best_manual_idx]
bands = assign_decision_band(
    best_manual_row['holdout_scores'],
    approve_threshold=0.25,
    reject_threshold=0.65,
)
pd.Series(bands).value_counts(normalize=True).rename('share')

## 9) Track 3: FLAML Optimization Lab

FLAML is run with explicit budget and PR-AUC objective to search for budget-aware challengers.

In [ ]:
flaml_result = run_flaml_track(
    X_train=X_train,
    y_train=y_train,
    X_holdout=X_holdout,
    y_holdout=y_holdout,
    time_budget=240,
    random_state=SEED,
)
flaml_result

## 10) Track 4: PyCaret Experiment Lab

PyCaret orchestration for compare/tune/calibrate/finalize workflow and deployability check.

In [ ]:
pycaret_result = run_pycaret_track(
    train_df=train_df,
    holdout_df=holdout_df,
    target_col='TARGET',
    session_id=SEED,
    model_output_path=ARTIFACT_DIR / 'pycaret_credit_risk',
)
pycaret_result

## 11) Unified Leaderboard and Final Model Ranking

Includes LazyPredict top candidates, manual top 3, FLAML winner, PyCaret finalized model, and business baseline.

In [ ]:
baseline_rows = [{
    'model_name': 'rule_based_income_ratio_baseline',
    'primary': 0.0,
    'secondary': 0.0,
    'tertiary': 0.25,
    'note': 'Business reference only; replace after scorecard benchmark run'
}]

leaderboard = build_leaderboard(
    project_name=PROJECT_NAME,
    task_type='binary_classification',
    lazy_results=eligible_table,
    manual_results=manual_results,
    flaml_result=flaml_result,
    pycaret_result=pycaret_result,
    baseline_rows=baseline_rows,
    manual_model_objects=manual_models,
)
leaderboard = rerank_with_business_weights(leaderboard)
leaderboard.head(10)

In [ ]:
leaderboard.to_csv(ARTIFACT_DIR / 'leaderboard_credit_risk.csv', index=False)
leaderboard[['model_name', 'library_source', 'rank_score', 'final_rank']].head(10)

## 12) Business Recommendation

Document in this section after execution:
- Why rank-1 is preferred for production
- When rank-2 may be safer (stability/latency/interpretability)
- Which tradeoff prevented another strong model from winning

## 13) Inference / Deployment Path

Persist chosen model + preprocessor + thresholds for FastAPI scoring.

In [ ]:
winner = leaderboard.sort_values('final_rank').iloc[0]['model_name']
if winner in manual_models:
    save_inference_bundle(
        model=manual_models[winner],
        preprocessor=preprocessor,
        output_dir=ARTIFACT_DIR,
        approve_threshold=0.25,
        reject_threshold=0.65,
    )
    print(f'Saved manual winner: {winner}')
else:
    print('Winner not in manual track; save selected production artifact separately.')

## 14) Monitoring / Drift / Retraining Plan

Recommended production checks:
- PR-AUC, ROC-AUC, Brier on rolling windows
- band distribution drift (approve/review/reject)
- score PSI and feature drift
- manual-review override rate
- retraining SLA and champion/challenger cadence

## 15) Limitations and Next Steps

- Add external bureau updates for recency-aware features
- Stress test fairness and reject-inference strategy
- Add explainability slices by segment and approval band
- Run seed stability reruns for top 3 finalists